In [ ]:
import math
from pathlib import Path

import scipy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import model_selection, metrics
from lightgbm import LGBMClassifier, LGBMRegressor
import lightgbm as lgb

In [ ]:
from warnings import simplefilter
simplefilter("ignore", category=FutureWarning)

In [ ]:
%%html
<style>
table {float:left}
</style>

# Files

In [ ]:
df_train = pd.read_csv("/kaggle/input/playground-series-s4e4/train.csv")
print(f'{df_train.shape=}')

df_test = pd.read_csv("/kaggle/input/playground-series-s4e4/test.csv")
print(f'{df_test.shape=}')

df_original = pd.read_csv("/kaggle/input/abalone-dataset/abalone.csv")
print(f'{df_original.shape=}')

# Features 
cf. https://archive.ics.uci.edu/dataset/1/abalone

| Variable Name  | Role    | Type        | Description                 | Units | Missing Values |
| -------------- | ------- | ----------- | --------------------------- | ----- | -------------- |
| Sex            | Feature | Categorical | M, F, and I (infant)        | \-    | no             |
| Length         | Feature | Continuous  | Longest shell measurement   | mm    | no             |
| Diameter       | Feature | Continuous  | Perpendicular to length     | mm    | no             |
| Height         | Feature | Continuous  | With meat in shell          | mm    | no             |
| Whole_weight   | Feature | Continuous  | Whole abalone               | grams | no             |
| Shucked_weight | Feature | Continuous  | Weight of meat              | grams | no             |
| Viscera_weight | Feature | Continuous  | Gut weight (after bleeding) | grams | no             |
| Shell_weight   | Feature | Continuous  | After being dried           | grams | no             |
| Rings          | Target  | Integer     | +1.5 gives the age in years | \-    | no             |

In [ ]:
def get_summary(df):
    df_desc = pd.DataFrame(df.describe(include='all').transpose())
    df_summary = pd.DataFrame({
        'dtype': df.dtypes,
        '#missing': df.isnull().sum().values,
        '#duplicates': df.duplicated().sum(),
        '#unique': df.nunique().values,
        'min': df_desc['min'].values,
        'max': df_desc['max'].values,
        'avg': df_desc['mean'].values,
        'std dev': df_desc['std'].values,
    })
    return df_summary

get_summary(df_train.drop('id', axis=1)).style.background_gradient()

In [ ]:

get_summary(df_test.drop('id', axis=1)).style.background_gradient()

In [ ]:

get_summary(df_original).style.background_gradient()

* We don't have any missing values
* No duplicates
* One categorical feature (Sex), all other features are continuous
* Target is integer
* On first sight, no differences between train, test and original dataset, but we'll check on that below

In [ ]:
# modify original column names to match train/test
df_original.columns = df_train.drop('id', axis=1).columns

# make life easier for LGBM
df_train['Sex'] = df_train['Sex'].astype("category")  
df_test['Sex'] = df_test['Sex'].astype("category")  
df_original['Sex'] = df_original['Sex'].astype("category")  

# Adversarial Validation (Train vs Test, Train vs Original)

> **Adversarial validation**
> checks if training and test (and original) data come from the same distribution. We basically merge train, test, and original data and have a model predict the dataset. If it can do that, we have a problem.

In [ ]:
# let's merge train with test/original data; use dataset as target label
df_adv_train_test = pd.concat([
    df_train.drop(['id', 'Rings'], axis=1).assign(dataset='train'),
    df_test.drop('id', axis=1).assign(dataset='test'),
])
print(f'{df_adv_train_test.shape=}')

df_adv_train_original = pd.concat([
    df_train.drop(['id'], axis=1).assign(dataset='train'),
    df_original.assign(dataset='original'),
])
print(f'{df_adv_train_original.shape=}')

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    df_adv_train_test.drop('dataset', axis=1),
    df_adv_train_test['dataset'], 
    test_size=0.33, 
    random_state=42)

clf = LGBMClassifier()
clf.fit(X_train, y_train)

pred_proba = clf.predict_proba(X_test)
print(pred_proba.shape)
auc_score = metrics.roc_auc_score(y_test, pred_proba[:,1])
print(f'{auc_score=}')

* A ROC_AUC Score of 0.5014 basically means the model can't differ train from test at all. Both are obviously drawn from the same distribution.

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    df_adv_train_original.drop('dataset', axis=1),
    df_adv_train_original['dataset'], 
    test_size=0.33, 
    random_state=42)

clf = LGBMClassifier()
clf.fit(X_train, y_train)

pred_proba = clf.predict_proba(X_test)
print(pred_proba.shape)
auc_score = metrics.roc_auc_score(y_test, pred_proba[:,1])
print(f'{auc_score=}')

* A ROC_AUC Score of 0.5955 means there are some differences between train and original data. We can try using original data for our models but should be cautious.

# Feature Distribution

In [ ]:
categorical_columns = ['Sex']
metric_columns = ['Length', 'Diameter', 'Height', 'Whole weight', 'Whole weight.1', 'Whole weight.2', 'Shell weight']

## Categorical

In [ ]:
def plot_categorical_feature_distribution(ser: pd.Series) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes = axes.flatten()
    value_counts = ser.value_counts(ascending=True)
    labels = value_counts.index.tolist()
    colors = sns.color_palette("Set2",len(labels)).as_hex()  # we borrow colors from a seaborn color palette
    # Donut Chart
    axes[0].pie(
        value_counts, 
        autopct='%1.1f%%', 
        textprops={'size': 8, 'color': 'black'}, 
        colors=colors,
        wedgeprops=dict(width=0.4),  # donut wedge width
        startangle=80, 
        pctdistance=0.85  # have percentage displayed within wedge
    )
    # Count Plot 
    for i, v in enumerate(value_counts):
        axes[1].barh(y=i, width=v, color='none', edgecolor=colors[i], hatch='////')
        axes[1].text(x=v + 1, y=i, s=str(v), color='black', fontsize=10, va='center')
    axes[1].set_yticks(range(len(labels)))
    axes[1].set_yticklabels(labels)
    sns.despine(left=True, bottom=True)  # remove default spines (borders) from plot
    axes[1].set_xticks([])
    fig.suptitle(f'{ser.name} Distribution', fontsize=15)
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()
    
plot_categorical_feature_distribution(ser=df_train['Sex'])

## Metric

In [ ]:
def dist(df_features: pd.DataFrame, 
         df_features_2: pd.DataFrame | None = None, 
         ncols=4, 
         labels: tuple[str, str] = ('Train', 'Original'),
         hide_xlabel=True,
         hide_ylabel=True,
        ) -> None:
    assert df_features_2 is None or set(df_features.columns) == set(df_features_2.columns) 
    
    nrows = (math.ceil(df_features.shape[1]/ncols))
    fig, axes = plt.subplots(nrows=nrows, 
                            ncols=min(df_features.shape[1], ncols), 
                            figsize=(20, nrows*2 + 2))
    axes = axes.flatten() if df_features.shape[1] > 1 else [axes]
    
    for i, col in enumerate(df_features):
        sns.kdeplot(df_features[col], ax=axes[i], fill=True, alpha=0.5, linewidth=0.5, color='#058279', label='Train')
        median_features = df_features[col].median()
        axes[i].axvline(x=median_features, color='#4caba4', linestyle='--')
        if df_features_2 is not None:
            sns.kdeplot(df_features_2[col], ax=axes[i], fill=True, alpha=0.5, linewidth=0.5, color='#db8067', label='Original')
            title = f'{col}, {labels[0]} skewness: {df_features[col].skew():.2f}\n {labels[1]} skewness: {df_features_2[col].skew():.2f}'
            median_features_2 = df_features_2[col].median()
            axes[i].axvline(x=median_features_2, color='#d68c78', linestyle='--')
            axes[i].legend(labels=[labels[0], 'Median', labels[1]])
        else:
            title = f'{col}, skewness: {df_features[col].skew():.2f}'
            axes[i].legend(labels=[labels[0], 'Median'])
        axes[i].set_title(title)
        if hide_xlabel:
            axes[i].set_xlabel('')
            axes[i].set_xticks([])
        if hide_ylabel:
            axes[i].set_ylabel('')
            axes[i].set_yticks([])
        
    plt.tight_layout()
    sns.despine(left=True, bottom=True)
    [fig.delaxes(ax) for ax in axes if not ax.has_data()]
    
dist(df_features=df_train[metric_columns], 
     df_features_2=df_original[metric_columns]
    )

* The original dataset generally has a "smoother" distribution within metric columns than the artificial training dataset.
* All features seem to be more or less Gaussian

# Skewness
For positively skewed features (right skew), one might consider Log-Transformation.

In [ ]:
for col in metric_columns:
    skewness = scipy.stats.skew(df_train[col])
    print(f'{col :<14}: {skewness}')
    
right_skew_cols = [col for col in metric_columns 
                   if scipy.stats.skew(df_train[col])> 0]

In [ ]:
def plot_distribution_with_log_transform(df_features: pd.DataFrame):
    fig, axes = plt.subplots(nrows=df_features.shape[1],
                             ncols=2, 
                             figsize=(10, df_features.shape[1]*2.3 + 2),
                             sharey="row")

    for i, feature in enumerate(df_features.columns):
        sns.kdeplot(x=df_features[feature], 
                    ax=axes[i, 0], 
                    fill=True, 
                    alpha=0.5, 
                    linewidth=0.5, 
                    color='#058279')

        sns.kdeplot(np.log1p(df_features[feature].rename(f'{feature} (log)')), 
                    ax=axes[i, 1], 
                    fill=True, 
                    alpha=0.5, 
                    linewidth=0.5, 
                    color='#db8067')

    sns.despine(left=False, bottom=True)
    plt.tight_layout()
    plt.show()
    
plot_distribution_with_log_transform(df_train[right_skew_cols])

# Target

In [ ]:
print(f'Skewness Rings in Train   : {df_train["Rings"].skew() :.2f}')
print(f'Skewness Rings in Original: {df_original["Rings"].skew() :.2f}')

In [ ]:
sns.displot(df_adv_train_original, 
            x="Rings", 
            col="dataset", 
            stat='percent', 
            discrete=True,
            shrink=.8, 
            height=5, 
            common_norm=False)

* Gaussian, slightly more skewed in train dataset

# Correlation

In [ ]:
def draw_correlation_heatmap(df_features: pd.DataFrame, 
                             absolute=False, 
                             annot=False,  # display correlation values in cells
                            ) -> None:
    df_corr = df_features.corr(method='pearson')
    if absolute:
        df_corr = df_corr.abs()
    triangle_mask = np.triu(np.ones_like(df_corr))
    fig, ax = plt.subplots(figsize=(9, 8))
    sns.heatmap(df_corr, 
                mask=triangle_mask, 
                cmap=sns.color_palette("light:b", as_cmap=True) if absolute else 'RdYlBu_r', 
                cbar=True, 
                linewidth=1, 
                annot=annot, #fmt='.2f',
                ax=ax)
    ax.set_title('Pearson Correlation' + (' (absolute)' if absolute else ''), fontsize=16)
    plt.tight_layout()
    plt.show()

draw_correlation_heatmap(df_features=df_train[metric_columns + ['Rings']],
                         annot=True,
                        )

* We have an extremely high correlation between Length an Diameter; we'll try dropping one of them
* All Features are highly correlated
* Minor correlation between all features and Target ("Rings")

In [ ]:
def plot_dist_by_category(ser_values: pd.Series, ser_categories: pd.Series, title="") -> None:
    categories = ser_categories.unique()
    colors = sns.color_palette("Set2",len(categories)).as_hex()
    
    fig, axes = plt.subplots(nrows=len(categories),
                             ncols=1, 
                             figsize=(10, len(categories)*2.3 + 2))
    axes = axes.flatten()
    
    for (ax, category, color) in zip(axes, categories, colors):
        
        sns.histplot(x=ser_values[ser_categories == category],
                     stat='percent', 
                     discrete=True,
                     shrink=.8, 
                     common_norm=False,
                     ax=ax,
                     color=color)
        ax.set_title(category)

    sns.despine(left=False, bottom=True)
    plt.tight_layout()
    fig.suptitle(title, y=1.02)
    plt.show()
    
plot_dist_by_category(df_train['Rings'], df_train['Sex'], 'Rings by Sex')

# Outliers

In [ ]:
def plot_outliers(df_features: pd.DataFrame, ncols=4) -> None:
    nrows = (math.ceil(df_features.shape[1]/ncols))
    fig = plt.figure(figsize=[18, nrows*2+2])
    plt.suptitle('Outliers', fontsize=18, fontweight='bold')
    fig.subplots_adjust(top=0.92)
    fig.subplots_adjust(hspace=0.5, wspace=0.4)
    for i, feature in enumerate(df_features):
        ax = fig.add_subplot(nrows, ncols, i+1)
        ax = sns.boxplot(data=df_features, 
                         x=feature, 
                         palette='pastel')
        ax.set_title(f'{feature}')
        ax.set_xlabel('')
        ax.grid(False)
    plt.show()
plot_outliers(df_features=df_train[metric_columns])

* Probably no problems with outliers

# Baseline Model
Train a GBDT without any fine-tuning as a baseline. Save the OOF predictions in case we want to use the model in an ensemble to optimize weights or train meta model.

In [ ]:
ensemble = []
arr_oof_predictions = np.zeros(len(df_train))
kf = model_selection.KFold(
    n_splits=5, shuffle=True, random_state=42
)
for fold_no, (train_idx, val_idx) in enumerate(kf.split(df_train)):
    print(f"Starting fold {fold_no}")

    df_train_fold = df_train.drop(['id', 'Rings'], axis=1).iloc[train_idx]
    df_val_fold = df_train.drop(['id', 'Rings'], axis=1).iloc[val_idx]
    print(f'{df_train_fold.shape=}, {df_val_fold.shape=}')
    
    ser_targets_train_fold = df_train['Rings'].iloc[train_idx]
    ser_targets_val_fold = df_train['Rings'].iloc[val_idx]

    estimator = LGBMRegressor()
    estimator.fit(df_train_fold, ser_targets_train_fold)
    
    arr_oof_predictions[val_idx] = estimator.predict(df_val_fold)
    
    estimator.booster_.save_model(f"lgb_regressor_fold_{fold_no}.txt")

In [ ]:
rmsle = metrics.mean_squared_log_error(y_true=df_train['Rings'],
                                       y_pred=arr_oof_predictions,
                                       squared=False)  # True would get us the MSLE
print(f'{rmsle=}=')

In [ ]:
df_train_oof = df_train.copy().assign(oof_prediction=arr_oof_predictions)
df_train_oof.to_pickle("df_train_oof.pkl")

# Predict Test & Submit
We're reloading the models from disk as we would in an ensemble later.

In [ ]:
folder = Path("/kaggle/working/")
estimator_paths = list(folder.glob("lgb_regressor_fold_*"))
display(estimator_paths)

arr_predictions_list: list[np.ndarray] = []
    
for path in estimator_paths:
    estimator = lgb.Booster(model_file=path)
    arr_predictions_list.append(estimator.predict(df_test.drop(['id'], axis=1)))

arr_mean_predictions = np.mean(arr_predictions_list, axis=0)
print(f'{arr_mean_predictions.shape=}')
arr_mean_predictions

In [ ]:
df_submission = pd.DataFrame({'id': df_test['id'],
                              'Rings': arr_mean_predictions})
df_submission.to_csv('submission.csv',
                     index=False)

df_submission